In [49]:
import pandas as pd
import numpy as np
import glicko2
import copy
from benchmarks import log_loss
import math

#from fighters_fights.csv create a dataframe with fighter, date, glicko rating

def rating_func():

    folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/2. data_cleaning/'
    df = pd.read_csv(folder + 'fighters_fights.csv')
    df['opponent_id'] = df.groupby('fight_id')['fighter_id'].transform(lambda x: x[::-1].values)
    df = df.sort_values('date', ascending=True).drop_duplicates('fight_id').reset_index(drop=True)

    rating_dict = {} # [fighter, date]: glicko rating 
    initialized_fighters = set()

    for i, row in df.iterrows():
        fighter, opponent, date = row['fighter_name'], row['opponent_name'], row['date']
        if fighter not in initialized_fighters:
            rating_dict[(fighter, date)] = glicko2.Player()
            initialized_fighters.add(fighter)
        if opponent not in initialized_fighters:
            rating_dict[(opponent, date)] = glicko2.Player()
            initialized_fighters.add(opponent)

        last_date = max([d for (f, d) in rating_dict.keys() if f == fighter])
        opp_last_date = max([d for (f, d) in rating_dict.keys() if f == opponent])
        fighter_object = copy.deepcopy(rating_dict[(fighter, last_date)])
        opponent_object = copy.deepcopy(rating_dict[(opponent, opp_last_date)])

        # next two lines in order to freeze variables form objects; makes logic less messy and less variable tracking :)
        fighter_rating, fighter_rd = rating_dict[(fighter, last_date)].rating, rating_dict[(fighter, last_date)].rd
        opponent_rating, opponent_rd = rating_dict[(opponent, opp_last_date)].rating, rating_dict[(opponent, opp_last_date)].rd

        if fighter == row['winner_name']:
            fighter_object.update_player([opponent_rating], [opponent_rd], [1])
            rating_dict[(fighter, date)] = fighter_object
        
            opponent_object.update_player([fighter_rating], [fighter_rd], [0])
            rating_dict[(opponent, date)] = opponent_object
        else:
            fighter_object.update_player([opponent_rating], [opponent_rd], [0])
            rating_dict[(fighter, date)] = fighter_object
        
            opponent_object.update_player([fighter_rating], [fighter_rd], [1])
            rating_dict[(opponent, date)] = opponent_object


    df_ratings = pd.DataFrame(
        [(f, d, p.rating, p.rd, p.vol) for (f, d), p in rating_dict.items()],
        columns=['fighter', 'prior_to_date', 'rating', 'rating_deviation', 'volatility'])

    df_ratings = df_ratings.sort_values(['fighter', 'prior_to_date'], ascending = True)

    # pre-fight rating: the rating carried INTO each fight (previous fight's result)
    df_ratings['rating'] = df_ratings.groupby('fighter')['rating'].shift(1).fillna(1500)
    df_ratings['rating_deviation']     = df_ratings.groupby('fighter')['rating_deviation'].shift(1).fillna(350)
    df_ratings['volatility'] = df_ratings.groupby('fighter')['volatility'].shift(1).fillna(0.06)

    df_ratings.to_csv('ratings.csv', index = False)

def outcomes(df_fighters_fights):
    outcomes = [] #1 if fighter 1 won, 0 if not
    for i, row in df_fighters_fights.iterrows():
        fighter_1 = row['fighter_name']
        fighter_2 = row['opponent_name']

        if fighter_1 == row['winner_name']:
            outcomes.append(1)
        else:
            outcomes.append(0)
        
def prob(r1, d1, r2, d2):
    q = 173.7178
    mu1, mu2 = (r1 - 1500) / q, (r2 - 1500) / q
    phi1, phi2 = d1 / q, d2 / q
    g = 1 / math.sqrt(1 + 3 * (phi1**2 + phi2**2) / math.pi**2)
    prob = 1 / (1 + math.exp(-g * (mu1 - mu2)))
    return prob

def probabilities(df_ratings, df_fighters_fights):
    probabilities = []
    for i, row in df_fighters_fights.iterrows():
        fighter_1 = row['fighter_name']
        fighter_2 = row['opponent_name']
        date = row['date']
    
        row_1 = df_ratings.loc[(df_ratings['fighter'] == fighter_1) & (df_ratings['prior_to_date'] == date)]
        row_2 = df_ratings.loc[(df_ratings['fighter'] == fighter_2) & (df_ratings['prior_to_date'] == date)]

        f1_r, f1_d = row_1['rating'].iloc[0], row_1['rating_deviation'].iloc[0]
        f2_r, f2_d = row_2['rating'].iloc[0], row_2['rating_deviation'].iloc[0]
        probability = prob(f1_r, f1_d, f2_r, f2_d)
    
        probabilities.append(probability)
    return probabilities

#


In [ ]:
folder = '/Users/alejandrogomez-paz/Desktop/UFC Project/'
df1 = pd.read_csv(folder + '2. data_cleaning/fighters_fights.csv')
cols = ['round_finished', 'round_time_sec', 'round_total', 'round_time_sec', 'fighter_id', 'event', 'stoppage_time_sec', 'method_type', 'method_specific']
df1 = df1.drop(columns = cols)
df1 = df1[::2]
df1 = df1.sort_values('date', ascending = True)
df2 = pd.read_csv(folder + '3. model/ratings_only/ratings.csv')

,fight_id,winner_name,loser_name,opponent_name,fighter_name,date
19488,567a09fd200cfa05,Gerard Gordeau,Teila Tuli,Teila Tuli,Gerard Gordeau,1993-11-12
20800,2d2bbc86e941e05c,Kevin Rosier,Zane Frazier,Kevin Rosier,Zane Frazier,1993-11-12
19934,64139d1d505e46c5,Royce Gracie,Gerard Gordeau,Royce Gracie,Gerard Gordeau,1993-11-12
19008,46acd54cc0c905fb,Ken Shamrock,Patrick Smith,Patrick Smith,Ken Shamrock,1993-11-12
20596,00b0796724ec1c09,Jason DeLucia,Trent Jenkins,Trent Jenkins,Jason DeLucia,1993-11-12


In [ ]:
outcomes =(df1)
probabilities = probabilities(df2, df1)

tau = 0.5 #randomly initialized on the lower end of the recomended 0.3 - 1.2 range

glicko2.Player._tau = tau